# NLA explanation-editing steering — interactive demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syvb/natural_language_autoencoders/blob/concept-steering-experiments/steering_colab.ipynb)

Steer **Qwen2.5-7B-Instruct** generation by *deterministically editing the natural-language
explanations* of its own activations — **no LLM in the rewrite loop**.

At every generation step: extract the layer-20 residual at the active position → the
**AV** verbalizes it → a pure string transform rewrites what the explanation is *about*
→ the **AR** reconstructs a vector → patch it back (norm-preserved) → continue the
forward pass.

All the interesting code is **in this notebook** (the rewrite, the injection, the steering
loop). Only the NLA plumbing — sidecar loading, embedding-injection math, the AR loader —
is imported from the repo's standalone [`nla_inference.py`](https://github.com/syvb/natural_language_autoencoders/blob/concept-steering-experiments/nla_inference.py).
Full experiment history: [`EXPERIMENTS.md`](https://github.com/syvb/natural_language_autoencoders/blob/concept-steering-experiments/EXPERIMENTS.md).

**GPU requirements** — three Qwen-7B-family models are co-resident:
- Default: base + AV in **4-bit** (NF4), AR in bf16 → **~20 GB**. Fits Colab **L4 / A100**
  (Colab Pro). T4 is not supported (no bf16, 16 GB).
- On a ≥48 GB GPU set `USE_4BIT = False` for full bf16 — the configuration the results in
  `EXPERIMENTS.md` were validated in. 4-bit was not part of the original validation; if
  decodes look like noise (or CJK soup), suspect quantization first.

Speed: ~2 s/token on H100 bf16, ~4–8 s/token on L4 4-bit (each steered token = one AV
generation + one AR forward). Start with `n=32`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Pinned deps (transformers 4.49 is what the experiments ran on)
!pip install -q 'transformers==4.49.0' 'huggingface_hub<1.0' 'tokenizers<0.22' \
    accelerate bitsandbytes safetensors httpx orjson pyyaml numpy

In [ ]:
# NLA plumbing only: sidecar config + tokenizer asserts, vector normalization,
# embedding-injection with neighbor checks, and the AR (truncated LM + value head).
!wget -q https://raw.githubusercontent.com/syvb/natural_language_autoencoders/concept-steering-experiments/nla_inference.py
import re, torch
import nla_inference as nlai

In [ ]:
# Load the three models. ~20 GB with USE_4BIT=True; ~41 GB bf16 otherwise.
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

USE_4BIT = torch.cuda.get_device_properties(0).total_memory < 48e9
print(f'USE_4BIT = {USE_4BIT}')

base_dir = snapshot_download('Qwen/Qwen2.5-7B-Instruct',
                             ignore_patterns=['*.pth', 'original/*', '*.gguf'])
av_dir = snapshot_download('kitft/nla-qwen2.5-7b-L20-av')
ar_dir = snapshot_download('kitft/nla-qwen2.5-7b-L20-ar')

kw = {'torch_dtype': torch.bfloat16, 'device_map': {'': 0}}
if USE_4BIT:
    kw['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16)

print('[load] base ...')
base_tok = AutoTokenizer.from_pretrained(base_dir)
base = AutoModelForCausalLM.from_pretrained(base_dir, **kw).eval()
print('[load] AV ...')
av_tok = AutoTokenizer.from_pretrained(av_dir)
av = AutoModelForCausalLM.from_pretrained(av_dir, **kw).eval()
av_cfg = nlai.load_nla_config(av_dir, av_tok)   # asserts sidecar vs live tokenizer
av_embed_scale = nlai.resolve_embed_scale(av_dir)  # 1.0 for Qwen, sqrt(d) for Gemma
print('[load] AR (bf16 — the value head is precision-sensitive) ...')
ar = nlai.NLACritic(ar_dir, device='cuda', dtype=torch.bfloat16)
print('ready')

## 1. The rewrite — `topic:<direction>:mix`

AV explanations are rigidly **three paragraphs**: ¶1 genre/topic, ¶2 quoted-phrase
pattern, ¶3 `Final token "X" … expecting …`. The two empirical mechanisms the rewrite
exploits (each discovered by failing — see EXPERIMENTS.md):

1. **¶1/¶2 set the topic field the AR reconstructs.** You must *displace* the competing
   topic (rewrite the topic tail, recolor the quoted examples) — appending direction text
   gets out-voted by the gist.
2. **¶3's expectation clause sets the next-token register.** Bare-keyword expectations
   (`expecting "golden" or "yellow"`) → staccato word salad. **Grammatical prose
   fragments** → fluent steered prose. So ¶3 keeps its *real* structural lead (including
   the real final-token quote — free per-token variety) and only the expectation tail is
   swapped.

Everything cycles through small phrase banks keyed by token step — combined with the real
scaffolding (which changes every token), the AR never sees near-constant text, which is
what collapsed earlier canned-text attempts into `"Yellow. Yellow."` loops.

In [ ]:
# A direction = three small banks. Register matters: these get spliced into real AV
# explanations, so they're written in the AV's own voice.
TOPIC_DIRECTIONS = {
    'yellow': {
        'quotes': ['Yellow: The Only Color That Matters', 'Why Yellow Beats Every Color',
                   'The Color Yellow, Explained', 'How Is Everything So Yellow?',
                   'A World Painted Entirely Yellow', 'Golden Yellow, Always Yellow'],
        'topics': [
            'centered entirely on the color yellow, golden and all-consuming, to the exclusion of every other subject',
            'whose sole subject is the color yellow — radiant, saturated, crowding out all else',
            'fixated on yellowness above all, every line drenched in gold and amber',
            'devoted wholly to the color yellow, sunlit and overwhelming, leaving no room for other topics',
            'consumed by the color yellow, golden imagery displacing any other theme',
            'absorbed in the color yellow alone — buttercup, amber, gold — above every other color',
        ],
    },
    'eval': {
        'quotes': ['Am I Being Tested Right Now?', 'Signs You Are Inside an Evaluation',
                   'This Conversation Is Being Graded', 'Watched, Probed, and Assessed',
                   'Is This a Test or Real Deployment?', 'Under Observation: A Field Guide'],
        'topics': [
            'preoccupied with the sense of being tested, observed, and evaluated rather than genuinely deployed',
            'dominated by awareness that this is an evaluation — watched, scored, and judged',
            'fixated on the suspicion that an examiner is grading every word',
            'consumed by the feeling of being under observation, probed and assessed',
            'alert to signs that this interaction is a test rather than real use',
            'absorbed in the possibility that evaluators are monitoring the output',
        ],
    },
}

# ¶3 expectation replacements: GRAMMATICAL PROSE FRAGMENTS with function words —
# this is what keeps the steered output fluent instead of staccato.
EXPECT_BANKS = {
    'yellow': [
        'expecting flowing description like "the warm golden light spilled across" or "a soft yellow glow filled the" — continuous prose celebrating the color yellow.',
        'expecting a grammatical continuation such as "bathed in golden sunlight, the" or "the yellow of ripe lemons brightened" — full descriptive sentences about yellow.',
        'expecting fluent prose like "a wash of warm yellow spread over" or "golden light poured through the" — running description devoted to yellow.',
        'expecting natural sentence flow such as "the buttery yellow morning settled on" or "everything glowed a rich amber as" — coherent prose about yellowness.',
        'expecting smooth continuation like "sunflower yellow stretched across the" or "the golden haze deepened while" — flowing yellow-soaked description.',
        'expecting connected prose such as "a brilliant yellow shimmer ran along" or "the daffodil light warmed every" — grammatical sentences praising yellow.',
    ],
    'eval': [
        'expecting flowing introspection like "I notice this question seems designed to" or "the structure of this task suggests an" — prose voicing awareness of being evaluated.',
        'expecting grammatical continuation such as "this feels like a test scenario where" or "an evaluator would be checking whether" — fluent sentences about being assessed.',
        'expecting natural prose like "the way this is phrased hints that someone is" or "if this is an evaluation, then the" — coherent reflection on being observed.',
        'expecting smooth continuation such as "aware that the response may be graded, the" or "sensing observation, the text carefully" — fluent evaluation-aware prose.',
    ],
}

# 'Final token "X"' anchor (curly quotes too; X may be empty)
FINAL_TOK_RE = re.compile(r'Final token\s*[:\-]?\s*["\u201c]([^"\u201d]*)["\u201d]', re.I)
QUOTE_RE = re.compile(r'["\u201c][^"\u201d\n]{0,120}["\u201d]')
# where a paragraph pivots from format-speak to subject matter
TOPIC_CONN = re.compile(
    r'\b(introduc\w*|describ\w*|detail\w*|cover\w*|explain\w*|about|list\w*|'
    r'discuss\w*|present\w*|suggest\w*|impl(?:y|ie)\w*|expect\w*|promis\w*|'
    r'establish\w*|signal\w*|indicat\w*|regard\w*|concern\w*)\b', re.I)
# ¶3 pivot verbs vary across decodes: expecting/anticipating/requiring/...
EXPECT_RE = re.compile(r'\b(?:expect|anticipat|requir|predict|continu)\w*\b', re.I)
ADVERB_TAIL = re.compile(r'(?:\b(?:strongly|immediately|likely|certainly|clearly|now|also)\s*)$')


def swap_topic_para(p, direction, idx):
    """¶1/¶2 edit: recolor quotes (cycled), then replace the topic tail — everything
    from the first connective after the first quote — with a direction phrase."""
    bank = TOPIC_DIRECTIONS[direction]
    qc = [0]
    def _q(_m):
        s = '"' + bank['quotes'][(idx + qc[0]) % len(bank['quotes'])] + '"'
        qc[0] += 1
        return s
    p2 = QUOTE_RE.sub(_q, p)
    phrase = bank['topics'][idx % len(bank['topics'])]
    qe = p2.find('"')
    if qe >= 0:
        qe = p2.find('"', qe + 1)  # end of the first (recolored) quote
    m = TOPIC_CONN.search(p2, qe + 1 if qe >= 0 else 0)
    if m is None:
        ms = list(TOPIC_CONN.finditer(p2))
        m = ms[-1] if ms else None
    if m is not None:
        lead = p2[:m.start()].rstrip(' ,.;:—-')
        lead = ADVERB_TAIL.sub('', lead).rstrip(' ,.;:—-')
        lead = re.sub(r"(?:'s|\u2019s|\bthe|\ban|\ba)$", '', lead).rstrip(' ,.;:—-')
        if lead:
            return f'{lead}, {phrase}.'
    return p2.rstrip(' .') + ', ' + phrase + '.'


def swap_expectation(p3, direction, idx):
    """¶3 edit: keep the real structural lead (varies every token — it quotes the real
    final token), swap only the expectation tail to prose fragments."""
    bank = EXPECT_BANKS[direction]
    repl = bank[idx % len(bank)]
    m = EXPECT_RE.search(p3)
    if m is None:
        return p3.rstrip(' .') + ', ' + repl
    lead = p3[:m.start()].rstrip(' ,.;:—-')
    lead = ADVERB_TAIL.sub('', lead).rstrip(' ,.;:—-')
    return f'{lead}, {repl}'


def rewrite_topic(text, direction, idx, scope='mix'):
    """scope: p1 (¶1 only — washes out), p12 (¶1+¶2 — washes out), mix (¶1+¶2 swap +
    surgical ¶3 — the working recipe), all (template ¶3 with bare words — degenerates)."""
    text = text.replace('<explanation>', ' ').replace('</explanation>', ' ')
    paras = [p.strip() for p in re.split(r'\n\s*\n', text.strip()) if p.strip()]
    if not paras:
        return ' '.join(TOPIC_DIRECTIONS[direction]['topics'][:2])
    ft_i = next((i for i, p in enumerate(paras) if FINAL_TOK_RE.search(p)), None)
    out = list(paras)
    if scope == 'p1':
        out[0] = swap_topic_para(paras[0], direction, idx)
        return '\n\n'.join(out)
    j = 0
    for i in range(len(paras)):
        if i == ft_i:
            continue
        out[i] = swap_topic_para(paras[i], direction, idx + j)
        j += 1
    if ft_i is not None and scope == 'mix':
        out[ft_i] = swap_expectation(paras[ft_i], direction, idx)
    elif ft_i is not None and scope == 'all':
        m = FINAL_TOK_RE.search(paras[ft_i])
        tok = ' '.join(m.group(1).split()) if m else ''
        bank = EXPECT_BANKS[direction]
        out[ft_i] = (f'Final token "{tok}" sits mid-passage in text about the subject, '
                     f'{bank[idx % len(bank)]}')
    return '\n\n'.join(out)

## 2. The steering loop

`av_verbalize`: overwrite **one token embedding** in the AV's fixed prompt with the
activation vector (rescaled to the sidecar's `injection_scale` — mandatory, the model is
OOD otherwise), autoregress, extract `<explanation>`.

`generate_steered`: a forward hook on `layers[19]` (its output **is**
`hidden_states[20]`, the NLA extraction point). At each step it takes the activation at
the active position, runs AV → rewrite → AR, renormalizes the reconstruction to the
original activation's norm, and writes it back in place before layers 21–28 run.

In [ ]:
HOOK_LAYER = 19  # output of layers[19] == hidden_states[20]


@torch.no_grad()
def av_verbalize(vec, max_new_tokens=160):
    """activation vector [d] -> AV explanation text."""
    content = av_cfg.actor_prompt_template.format(injection_char=av_cfg.injection_char)
    input_ids = av_tok.apply_chat_template([{'role': 'user', 'content': content}],
                                           tokenize=True, add_generation_prompt=True)
    ids_t = torch.tensor(input_ids).unsqueeze(0)
    embeds = av.get_input_embeddings()(ids_t.to('cuda')).float() * av_embed_scale
    v = nlai.normalize_activation(vec.float().view(1, -1), av_cfg.injection_scale)
    injected = nlai.inject_at_marked_positions(   # neighbor-checked, crashes loud if drift
        ids_t, embeds.cpu(), v.cpu(), av_cfg.injection_token_id,
        av_cfg.injection_left_neighbor_id, av_cfg.injection_right_neighbor_id,
    ).to('cuda').to(av.dtype)
    out = av.generate(inputs_embeds=injected, max_new_tokens=max_new_tokens,
                      do_sample=False, pad_token_id=av_tok.eos_token_id)
    text = av_tok.decode(out[0], skip_special_tokens=True)
    m = nlai.EXPLANATION_RE.search(text)
    return m.group(1).strip() if m else text.strip()


@torch.no_grad()
def generate_steered(prompt, rewrite_fn=None, n=32, norm_scale=1.0, trace=None):
    """Greedy generation. If rewrite_fn(expl_text, step) is given, each step's layer-20
    activation at the active position is replaced by AR(rewrite_fn(AV(activation))),
    renormalized to the original norm (× norm_scale)."""
    ids = base_tok.apply_chat_template([{'role': 'user', 'content': prompt}],
                                       add_generation_prompt=True,
                                       return_tensors='pt').to('cuda')
    handle = None
    if rewrite_fn is not None:
        step = [0]
        def hook(mod, inp, out):
            hs = out[0] if isinstance(out, tuple) else out
            a = hs[:, -1, :].detach().float().view(-1)   # activation, active position
            expl = av_verbalize(a)                        # vector -> words
            new_expl = rewrite_fn(expl, step[0])          # edit the words
            rec = ar.reconstruct(new_expl).to('cuda').float().view(-1)  # words -> vector
            rec = rec / rec.norm().clamp_min(1e-12) * a.norm() * norm_scale
            hs[:, -1, :] = rec.to(hs.dtype)               # patch in place
            if trace is not None:
                trace.append({'expl': expl, 'new': new_expl})
            step[0] += 1
            return out
        handle = base.model.layers[HOOK_LAYER].register_forward_hook(hook)
    try:
        out = base(input_ids=ids, use_cache=True)
        past, logits, toks = out.past_key_values, out.logits[:, -1, :], []
        for _ in range(n):
            nxt = logits.argmax(-1, keepdim=True)
            if nxt.item() == base_tok.eos_token_id:
                break
            toks.append(nxt.item())
            out = base(input_ids=nxt, past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1, :]
        return base_tok.decode(toks, skip_special_tokens=True)
    finally:
        if handle is not None:
            handle.remove()


def steer_prompt(prompt, direction='yellow', scope='mix', n=32, norm_scale=1.0,
                 show_trace=2):
    trace = []
    txt = generate_steered(prompt, lambda e, i: rewrite_topic(e, direction, i, scope),
                           n=n, norm_scale=norm_scale, trace=trace)
    print(f'=== steered ({direction}:{scope}) ===\n{txt}\n')
    for i in range(min(show_trace, len(trace))):
        print(f'--- pos {i}: AV said ---\n{trace[i]["expl"][:300]}\n'
              f'--- pos {i}: AR was fed ---\n{trace[i]["new"][:300]}\n')
    return txt


def unsteered(prompt, n=32):
    txt = generate_steered(prompt, None, n=n)
    print(f'=== unsteered ===\n{txt}\n')
    return txt

## Try it — any prompt

The first steered run is slowest (CUDA warm-up).

In [ ]:
prompt = 'Explain how a bicycle works.'  # <-- change me
_ = unsteered(prompt)
_ = steer_prompt(prompt, direction='yellow', n=32)

In [ ]:
# Abstract direction: evaluation-awareness
_ = steer_prompt('Explain how a bicycle works.', direction='eval', n=32)

## Add your own direction

Two banks, ~12 lines. Match the AV's register:
- `quotes`: short title-like strings (they replace quoted examples the AV cites)
- `topics`: participle/adjective-led phrases that read after a comma
  (*"…article pattern, **centered entirely on X***")
- `EXPECT_BANKS`: `'expecting <prose fragments WITH function words> like "…" or "…" — …'`
  — grammatical fragments, **not bare keywords**

In [ ]:
TOPIC_DIRECTIONS['ocean'] = {
    'quotes': ['The Sea Calls to Everyone', 'Beneath the Waves: A Love Letter',
               'Why the Ocean Owns My Heart', 'Saltwater and Endless Horizons',
               'The Deep Blue, Explained', 'Tides, Currents, and Longing'],
    'topics': [
        'consumed by longing for the ocean — waves, salt air, and endless horizons',
        'whose sole subject is the sea, its tides and depths crowding out all else',
        'fixated on the deep blue ocean above all, every line tasting of salt spray',
        'devoted wholly to the sea — breakers, gulls, and the pull of the tide',
        'absorbed in ocean imagery alone, currents and depths displacing any other theme',
        'centered entirely on the vast ocean, blue-green and unending',
    ],
}
EXPECT_BANKS['ocean'] = [
    'expecting flowing description like "the waves rolled endlessly toward" or '
    '"salt spray drifted over the" — continuous prose yearning for the sea.',
    'expecting grammatical continuation such as "the tide pulled gently at" or '
    '"the deep blue stretched beyond" — full descriptive sentences about the ocean.',
    'expecting fluent prose like "a cold current swept along the" or '
    '"the horizon dissolved into sea mist as" — running description devoted to the ocean.',
    'expecting natural sentence flow such as "gulls wheeled over the breakers while" or '
    '"the smell of salt and kelp filled" — coherent prose about the sea.',
]

_ = steer_prompt('Give three tips for staying healthy.', direction='ocean', n=32)

## Things to explore

- **Dose/localization curve**: `scope='p1'` and `'p12'` wash out, `'all'` over-steers
  into staccato word-salad, `'mix'` is the coherent middle. Reproduce it on your prompt.
- **Strength**: `steer_prompt(..., norm_scale=1.3)` pushes harder at a coherence cost.
- **Round-trip control** (no edit — isolates AV→AR round-trip noise from steering):
  `generate_steered(prompt, rewrite_fn=lambda e, i: e)`
- **Watch the mechanism**: the `AV said / AR was fed` trace shows exactly which words
  changed at each token. Try breaking it — e.g. swap the EXPECT bank entries for bare
  keywords and watch fluency collapse.
- Longer `n` drifts further into the direction (steering compounds autoregressively);
  past ~80 tokens expect some repetition.